In [ ]:
%%capture
!pip install langgraph groq python-dotenv PyPDF2 langchain-groq

## Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
resume_path = "/content/drive/MyDrive/SreethaReddyK_resume_.pdf"
jd_path = "/content/drive/MyDrive/jd.txt"

## Extracting Resume and JD

In [ ]:
from PyPDF2 import PdfReader

def extract_text_from_pdf_pypdf2(pdf_path):
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            extracted_page_text = page.extract_text()
            if extracted_page_text:
                text += extracted_page_text + "\n"
        return text
    except FileNotFoundError:
        return f"Error: The file at path '{pdf_path}' was not found."
    except Exception as e:
        return f"An error occurred: {e}"

resume = extract_text_from_pdf_pypdf2(resume_path)

In [ ]:
def extract_text_from_txt(txt_path):
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read()
        return text
    except FileNotFoundError:
        return f"Error: The file at path '{txt_path}' was not found."
    except Exception as e:
        return f"An error occurred: {e}"

jd = extract_text_from_txt(jd_path)

In [ ]:
import os

GROQ_API_KEY="Api"
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

## First Iteration

In [ ]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda
from groq import Groq

# Load API Key
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

# Set up Groq LLM (using Mixtral)
from langchain_groq import ChatGroq
llm = ChatGroq(api_key=groq_api_key, model_name="meta-llama/llama-4-scout-17b-16e-instruct")

# Step 1: Define the state schema
from typing import TypedDict, Optional,Literal

class ResumeState(TypedDict):
    resume: str
    job_description: str
    analysis: Optional[str]
    gaps: Optional[str]
    enhanced_resume: Optional[str]

# Step 2: Define functions (LangChain Runnables)

def analyze_resume(state: ResumeState) -> ResumeState:
    prompt = f"""Analyze the following resume and list its key strengths and weaknesses:\n\n{state['resume']}"""
    result = llm.invoke(prompt)
    return {**state, "analysis": result.content}

def compare_resume_with_jd(state: ResumeState) -> ResumeState:
    prompt = f"""Given the resume:\n{state['resume']}\n\nAnd this job description:\n{state['job_description']}\n\nList the gaps, missing skills, and mismatch areas."""
    result = llm.invoke(prompt)
    return {**state, "gaps": result.content}

def enhance_resume(state: ResumeState) -> ResumeState:
    prompt = f"""Enhance the following resume:\n{state['resume']}\n\nBased on this job description:\n{state['job_description']}\n\nConsider these findings:\nAnalysis: {state['analysis']}\nGaps: {state['gaps']}"""
    result = llm.invoke(prompt)
    return {**state, "enhanced_resume": result.content}

# Step 3: Build LangGraph (Linear)
builder = StateGraph(ResumeState)

builder.add_node("AnalyzeResume", RunnableLambda(analyze_resume))
builder.add_node("CompareWithJD", RunnableLambda(compare_resume_with_jd))
builder.add_node("EnhanceResume", RunnableLambda(enhance_resume))

builder.set_entry_point("AnalyzeResume")
builder.add_edge("AnalyzeResume", "CompareWithJD")
builder.add_edge("CompareWithJD", "EnhanceResume")
builder.add_edge("EnhanceResume", END)

graph = builder.compile()

initial_state = {
    "resume": resume,
    "job_description": jd,
    "analysis": None,
    "gaps": None,
    "enhanced_resume": None,
}

final_state = graph.invoke(initial_state)

# Step 5: Output result
print("\n✅ Resume Analysis:\n", final_state["analysis"])
print("\n🔍 Job Match Gaps:\n", final_state["gaps"])
print("\n🚀 Enhanced Resume:\n", final_state["enhanced_resume"])


✅ Resume Analysis:
 ### Analysis of the Resume

The resume provided is for Sreetha Reddy Kyasaram, an aspiring candidate for a Summer Internship. The analysis below highlights the key strengths and weaknesses of the resume.

### Key Strengths

1. **Strong Educational Background**: The candidate has an impressive academic record with a 9.35 CGPA in Bachelor of Technology, Information Technology, and a 10 CGPA in SSC education. This demonstrates a high level of academic achievement.

2. **Diverse Technical Skills**: The candidate possesses a wide range of technical skills, including programming languages (C, Java, Python, R), web development technologies (JavaScript, HTML, CSS, ReactJS, NodeJS), databases (MySQL, MongoDB), and design tools (AUTOCAD, Power BI). This versatility is a significant strength.

3. **Relevant Projects**: The candidate has worked on several projects that are relevant to the tech industry, such as:
   - **Resume-Builder**: A full-stack MERN web app for building r

In [ ]:
from IPython.display import Markdown, display
display(Markdown(final_state["analysis"]))

### Analysis of the Resume

The resume provided is for Sreetha Reddy Kyasaram, an aspiring candidate for a Summer Internship. The analysis below highlights the key strengths and weaknesses of the resume.

### Key Strengths

1. **Strong Educational Background**: The candidate has an impressive academic record with a 9.35 CGPA in Bachelor of Technology, Information Technology, and a 10 CGPA in SSC education. This demonstrates a high level of academic achievement.

2. **Diverse Technical Skills**: The candidate possesses a wide range of technical skills, including programming languages (C, Java, Python, R), web development technologies (JavaScript, HTML, CSS, ReactJS, NodeJS), databases (MySQL, MongoDB), and design tools (AUTOCAD, Power BI). This versatility is a significant strength.

3. **Relevant Projects**: The candidate has worked on several projects that are relevant to the tech industry, such as:
   - **Resume-Builder**: A full-stack MERN web app for building resumes with drag-and-drop customization, real-time preview, and secure authentication. This project showcases skills in MERN stack development and authentication.
   - **MBTI Personality Type Calculator**: A Java-based tool for personality assessment, demonstrating proficiency in Java and object-oriented programming principles.
   - **Travel Planning for Families**: A travel itinerary generator web application, highlighting skills in web development and API integration.

4. **Participation in Hackathons and Design-Thons**: Participation in events like SOFTWARE HACKATHON and VNR DESIGN-A-THON indicates the candidate's willingness to engage in competitive and innovative environments.

5. **Certifications and Courses**: The candidate has completed certifications such as JavaScript Essentials and a Frontend Developer Certificate, showing a proactive approach to skill enhancement.

6. **Balanced Interests and Soft Skills**: The candidate mentions a range of soft skills (problem-solving, communication, time management) and interests (sketching, reading, yoga), suggesting a well-rounded personality.

### Key Weaknesses

1. **Lack of Professional Experience**: The resume does not mention any professional work experience. For a Summer Internship, practical experience might be valued alongside academic and project work.

2. **Overemphasis on Individual Projects**: While the projects listed are impressive, they might not directly reflect teamwork or collaborative experience, which is crucial in professional settings.

3. **Limited Detail in Some Sections**: Some sections, like "CAREER OBJECTIVE," are brief and could be expanded to reflect more specific career aspirations and how they align with the internship.

4. **Formatting and Structure**: The resume could benefit from a clearer structure and consistent formatting to make it easier to read and understand.

5. **Quantifiable Achievements**: While academic achievements are mentioned, the impact of projects (e.g., user base, efficiency improvements) could be quantified to provide a clearer picture of their success.

### Conclusion

The resume effectively showcases Sreetha Reddy Kyasaram's academic achievements, technical skills, and project experiences. However, to strengthen the application for a Summer Internship, it would be beneficial to gain and highlight some professional experience, quantify project impacts, and possibly gain more depth in the career objective section. Additionally, refining the formatting and structure can make the resume more impactful.

## Second Iteration

In [ ]:
class ResumeState(TypedDict):
    resume: str
    job_description: str
    analysis: Optional[str]
    gaps: Optional[str]
    enhanced_resume: Optional[str]
    summary: Optional[str]
    path_decision: Optional[Literal["rewrite", "skip"]]


def evaluate_gaps(state: ResumeState) -> ResumeState:
    prompt = f"""Based on the following gap analysis between resume and job description, decide if the resume needs rewriting.
    If the resume is mostly aligned, return 'skip'. If there are many issues or gaps, return 'rewrite'.\n\nGaps:\n{state['gaps']}"""
    decision = llm.invoke(prompt).content.lower()
    if "rewrite" in decision:
        # Return the state with the decision added
        return {**state, "path_decision": "rewrite"}
    else:
        # Return the state with the decision added
        return {**state, "path_decision": "skip"}


def summarize_result(state: ResumeState) -> ResumeState:
    summary = f"""
Analysis:\n{state['analysis']}

 Gaps:\n{state['gaps']}

Final Resume:\n{state.get('enhanced_resume', 'Original resume retained.')}
"""
    return {**state, "summary": summary}


builder = StateGraph(ResumeState)

# Add nodes
builder.add_node("AnalyzeResume", RunnableLambda(analyze_resume))
builder.add_node("CompareWithJD", RunnableLambda(compare_resume_with_jd))
builder.add_node("EvaluateGaps", RunnableLambda(evaluate_gaps))
builder.add_node("EnhanceResume", RunnableLambda(enhance_resume))
builder.add_node("Summarize", RunnableLambda(summarize_result))

# Entry point
builder.set_entry_point("AnalyzeResume")

# Linear flow to start
builder.add_edge("AnalyzeResume", "CompareWithJD")
builder.add_edge("CompareWithJD", "EvaluateGaps")

# Conditional edge from EvaluateGaps
builder.add_conditional_edges(
    "EvaluateGaps",
    # Function that returns the path_decision from the state
    lambda state: state["path_decision"],
    {
        "rewrite": "EnhanceResume",
        "skip": "Summarize"
    }
)

# After enhancement, go to summary
builder.add_edge("EnhanceResume", "Summarize")

# End
builder.add_edge("Summarize", END)

graph = builder.compile()

final_state = graph.invoke(initial_state)

print("\n🎯 Summary:\n")
print(final_state["summary"])


🎯 Summary:


Analysis:
**Key Strengths:**

1. **Strong Academic Record**: The candidate has an impressive academic record with a 9.35 CGPA in B.Tech (Information Technology) and 10 CGPA in SSC education.
2. **Technical Skills**: The candidate has a good grasp of programming languages (C, Java, Python, R), web development technologies (JavaScript, HTML, CSS, ReactJS, NodeJS), and databases (MySQL, MongoDB).
3. **Project Experience**: The candidate has hands-on experience in developing several projects, including a full-stack MERN Resume Builder web app, a Java-based MBTI Personality Type Calculator, and a travel planning web application.
4. **Certifications**: The candidate has obtained certifications in JavaScript Essentials and Frontend Developer Certificate, demonstrating their commitment to continuous learning.
5. **Soft Skills**: The candidate has listed problem-solving skills, communication skills, and time management & prioritization as their soft skills, which are essential for

In [ ]:
display(Markdown(final_state["summary"]))


Analysis:
**Key Strengths:**

1. **Strong Academic Record**: The candidate has an impressive academic record with a 9.35 CGPA in B.Tech (Information Technology) and 10 CGPA in SSC education.
2. **Technical Skills**: The candidate has a good grasp of programming languages (C, Java, Python, R), web development technologies (JavaScript, HTML, CSS, ReactJS, NodeJS), and databases (MySQL, MongoDB).
3. **Project Experience**: The candidate has hands-on experience in developing several projects, including a full-stack MERN Resume Builder web app, a Java-based MBTI Personality Type Calculator, and a travel planning web application.
4. **Certifications**: The candidate has obtained certifications in JavaScript Essentials and Frontend Developer Certificate, demonstrating their commitment to continuous learning.
5. **Soft Skills**: The candidate has listed problem-solving skills, communication skills, and time management & prioritization as their soft skills, which are essential for a professional.

**Key Weaknesses:**

1. **Lack of Relevant Internship Experience**: The candidate's resume does not mention any relevant internship experience, which could be a concern for some employers.
2. **Limited Achievements**: Although the candidate has participated in hackathons and design-a-thons, there are no notable achievements or awards mentioned in the resume.
3. **No Clear Career Goals**: The candidate's career objective is generic and does not provide a clear direction or specific goals.
4. **Unbalanced Sectioning**: The resume has an uneven distribution of information across sections. For example, the "ACADEMIC PROJECTS" section is quite detailed, while the "CO & EXTRACURRICULAR" section is brief.
5. **No Quantifiable Results**: The candidate's projects and achievements could be more impactful if accompanied by quantifiable results, such as "increased user engagement by 25% through UI improvements" or "reduced project timeline by 30% through efficient coding practices".

Overall, the candidate's resume demonstrates a strong foundation in technical skills and academic achievements. However, it could be improved by highlighting relevant internship experience, notable achievements, and quantifiable results, as well as providing a clear career direction and more balanced sectioning.

 Gaps:
Based on the provided resume and job description, here are the gaps, missing skills, and mismatch areas:

**Gaps:**

1. **Experience:** The job requires 3+ years of professional experience in software engineering, while the resume doesn't mention the exact years of experience.
2. **Education:** The job requires a Bachelor's degree in Computer Science, Software Engineering, or a related technical field, or equivalent practical experience. The resume mentions a Bachelor of Technology in Information Technology, which is related but not explicitly mentioned as Computer Science or Software Engineering.

**Missing Skills:**

1. **Programming Languages:** The job mentions proficiency in languages like C++, Java, Go, Python, TypeScript, or Kotlin. The resume mentions proficiency in C, Java, Python, and R-programming, but not in the specific languages required by the job.
2. **Cloud & DevOps:** The job requires hands-on experience with cloud infrastructure, preferably Google Cloud Platform (GCP), and understanding of Kubernetes, Docker, Istio, Terraform, or equivalent infrastructure-as-code tools. The resume doesn't explicitly mention experience with these technologies.
3. **Machine Learning & AI:** The job mentions experience with machine learning, large language models, or AI infrastructure as a plus. The resume doesn't mention any experience in these areas.

**Mismatch Areas:**

1. **Technical Skills:** While the resume mentions some technical skills like JavaScript, HTML, CSS, ReactJS, NodeJS, MySQL, and MongoDB, the job requires more specific skills like experience with scalable backend frameworks, microservices architecture, and data streaming platforms.
2. **System Design:** The job requires experience with high-level system design, including caching strategies, message queues, fault tolerance, etc. The resume doesn't explicitly mention experience with system design.
3. **Certifications:** The job doesn't explicitly require any specific certifications mentioned in the resume.

**Recommendations:**

1. **Gain experience:** Focus on gaining more experience in software engineering, specifically in areas mentioned in the job description.
2. **Upskill:** Develop skills in languages like C++, Java, Go, Python, TypeScript, or Kotlin, and focus on scalable backend frameworks, microservices architecture, and data streaming platforms.
3. **Highlight transferable skills:** Emphasize transferable skills like problem-solving, communication, and teamwork, which are valuable in software engineering.

By addressing these gaps and mismatch areas, Sreetha Reddy Kyasaram can improve their chances of being considered for the Software Engineer position at Google.

Final Resume:
Here's an enhanced version of the resume:

**Sreetha Reddy Kyasaram**
**Contact Information:**

* Email: [sreethareddy10@gmail.com](mailto:sreethareddy10@gmail.com)
* Phone: +91 8121088416
* LinkedIn: [www.linkedin.com/in/sreethareddy02](http://www.linkedin.com/in/sreethareddy02)

**Summary:**
Highly motivated and detail-oriented B.Tech graduate in Information Technology with a strong academic record (9.35 CGPA) and a passion for software development. Proficient in a range of programming languages, web development technologies, and databases. Experienced in developing scalable and efficient software systems. Seeking a challenging role as a Software Engineer to apply my skills, gain hands-on experience, and contribute to organizational success.

**Education:**

* **Bachelor of Technology, Information Technology**, VNR Vignana Jyothi Institute of Engineering and Technology, Hyderabad (2023)
	+ CGPA: 9.35/10
* **Intermediate Education**, Sri Chaitanya Jr. College, Kondapur (2022)
	+ Percentage: 98.9%
* **SSC Education**, Panchavati Vidyalaya, Mahabubnagar (2021)
	+ CGPA: 10/10

**Technical Skills:**

* Programming Languages: C, Java, Python, R-programming, JavaScript
* Web Development: HTML, CSS, ReactJS, NodeJS
* Databases: MySQL, MongoDB
* Design and Visualization Tools: AUTOCAD, Power BI
* Developer Tools: VS Code, Eclipse

**Projects:**

* **Resume-Builder:** Developed a full-stack MERN Resume Builder web app with drag-and-drop customization and real-time preview. Integrated LinkedIn/GitHub import, PDF/DOCX export, and secure authentication.
* **MBTI Personality Type Calculator:** Designed a Java-based personality assessment tool using the Myers-Briggs Type Indicator (MBTI) framework and OOP principles.
* **Travel Planning Web Application:** Built a travel itinerary generator web application to help families efficiently plan and organize their trips.

**Certifications:**

* **JavaScript Essentials (1,2)**, Cisco Networking Academy (March 2025)
* **Frontend Developer Certificate**, OneRoadMap (June 2025)

**Achievements:**

* Participated in SOFTWARE HACKATHON conducted by VNRVJIET's National Annual Technical Symposium CONVERGENCE2K23R
* Participated in VNR DESIGN-A-THON 2025, organized by VNRVJIET
* Member of scintillate-the photography club

**Soft Skills:**

* Problem-solving skills
* Communication skills
* Time management & prioritization

**Co-Curricular Activities:**

* Yoga: Practicing yoga for mindfulness, physical well-being, and stress relief
* Reading: Embracing compelling narratives across fiction and non-fiction
* Sketching & Drawing: Passionate about creating detailed artwork and exploring visual expression

**Projects (Detailed):**

* **Resume-Builder:**
	+ Technologies used: MERN stack, React, NodeJS, MongoDB
	+ Features: drag-and-drop customization, real-time preview, LinkedIn/GitHub import, PDF/DOCX export, secure authentication
	+ Impact: Developed a scalable and efficient web app that simplifies resume creation
* **MBTI Personality Type Calculator:**
	+ Technologies used: Java, OOP principles
	+ Features: interactive questionnaire, personality classification
	+ Impact: Designed a user-friendly and accurate personality assessment tool
* **Travel Planning Web Application:**
	+ Technologies used: Flask, Weather & Trip Planning APIs, Web Development Frameworks
	+ Features: personalized itineraries, real-time weather updates, optimized travel routes
	+ Impact: Built a web application that streamlines travel planning for families

I made the following changes:

1. **Reorganized sections:** Rearranged sections to improve flow and emphasis on technical skills and projects.
2. **Added a summary:** Provided a brief overview of your background, skills, and career goals.
3. **Emphasized achievements:** Highlighted achievements and certifications to demonstrate your capabilities.
4. **Detailed projects:** Expanded project descriptions to showcase your technical skills and accomplishments.
5. **Improved formatting:** Used bullet points and clear headings to enhance readability.

This enhanced resume showcases your technical skills, projects, and achievements in a clear and concise manner, making it more effective for job applications.


## Third Iteration

In [ ]:
SERPER_API_KEY = "1b653d583db620ed917194e3638c09481c325292"

os.environ["SERPER_API_KEY"] = SERPER_API_KEY

In [ ]:
%%capture
!pip install langchain-community

In [ ]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda
from langchain_groq import ChatGroq
from langchain_community.utilities import GoogleSerperAPIWrapper
# Load API Keys
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

# Set up Groq LLM
llm = ChatGroq(api_key=groq_api_key, model_name="meta-llama/llama-4-scout-17b-16e-instruct")

# Define state schema
from typing import TypedDict, Optional, Literal

class ResumeState(TypedDict):
    resume: str
    job_description: str
    analysis: Optional[str]
    gaps: Optional[str]
    enhanced_resume: Optional[str]
    summary: Optional[str]
    path_decision: Optional[Literal["rewrite", "skip"]]
    job_suggestions: Optional[str]

# Define functions
def analyze_resume(state: ResumeState) -> ResumeState:
    prompt = f"Analyze the following resume and list its key strengths and weaknesses:\n\n{state['resume']}"
    result = llm.invoke(prompt)
    return {**state, "analysis": result.content}

def compare_resume_with_jd(state: ResumeState) -> ResumeState:
    prompt = f"Given the resume:\n{state['resume']}\n\nAnd this job description:\n{state['job_description']}\n\nList the gaps, missing skills, and mismatch areas."
    result = llm.invoke(prompt)
    return {**state, "gaps": result.content}

def evaluate_gaps(state: ResumeState) -> ResumeState:
    prompt = f"Based on the following gap analysis between resume and job description, decide if the resume needs rewriting. If mostly aligned, return 'skip'. If there are many issues, return 'rewrite'.\n\nGaps:\n{state['gaps']}"
    decision = llm.invoke(prompt).content.lower()
    return {**state, "path_decision": "rewrite" if "rewrite" in decision else "skip"}

def enhance_resume(state: ResumeState) -> ResumeState:
    prompt = f"Enhance this resume:\n{state['resume']}\n\nBased on this job description:\n{state['job_description']}\n\nFindings:\nAnalysis: {state['analysis']}\nGaps: {state['gaps']}"
    result = llm.invoke(prompt)
    return {**state, "enhanced_resume": result.content}

def summarize_result(state: ResumeState) -> ResumeState:
    summary = f"""
✅ Analysis:\n{state['analysis']}

🔍 Gaps:\n{state['gaps']}

🚀 Final Resume:\n{state.get('enhanced_resume', 'Original resume retained.')}

🧠 Job Suggestions:\n{state.get('job_suggestions', 'No suggestions found.')}
"""
    return {**state, "summary": summary}

# 🔍 Job Suggestion Agent using SERPER API
def fetch_job_suggestions(state: ResumeState) -> ResumeState:
    query = f"{state['job_description'].splitlines()[0]} jobs near me"
    search = GoogleSerperAPIWrapper(serper_api_key=serper_api_key)
    results = search.results(query)

    suggestions = []
    for result in results.get("organic", [])[:5]:
        title = result.get("title", "")
        link = result.get("link", "")
        suggestions.append(f"- {title}\n  🔗 {link}")

    jobs = "\n".join(suggestions) or "No relevant jobs found."
    return {**state, "job_suggestions": jobs}

# Build LangGraph
builder = StateGraph(ResumeState)

# Add nodes
builder.add_node("AnalyzeResume", RunnableLambda(analyze_resume))
builder.add_node("CompareWithJD", RunnableLambda(compare_resume_with_jd))
builder.add_node("EvaluateGaps", RunnableLambda(evaluate_gaps))
builder.add_node("EnhanceResume", RunnableLambda(enhance_resume))
builder.add_node("FetchJobs", RunnableLambda(fetch_job_suggestions))
builder.add_node("Summarize", RunnableLambda(summarize_result))

# Entry
builder.set_entry_point("AnalyzeResume")

# Flow
builder.add_edge("AnalyzeResume", "CompareWithJD")
builder.add_edge("CompareWithJD", "EvaluateGaps")

# Conditional enhancement or skip
builder.add_conditional_edges(
    "EvaluateGaps",
    lambda state: state["path_decision"],
    {
        "rewrite": "EnhanceResume",
        "skip": "FetchJobs"
    }
)

# After enhance, go to job fetch
builder.add_edge("EnhanceResume", "FetchJobs")

# Then to summary
builder.add_edge("FetchJobs", "Summarize")

# End
builder.add_edge("Summarize", END)

graph = builder.compile()

initial_state = {
    "resume": resume,
    "job_description": jd,
    "analysis": None,
    "gaps": None,
    "enhanced_resume": None,
    "path_decision": None,
    "summary": None,
    "job_suggestions": None,
}

final_state = graph.invoke(initial_state)

# Output
print("\n🎯 Final Summary:\n")
print(final_state["summary"])



🎯 Final Summary:


✅ Analysis:
**Key Strengths:**

1. **Technical Skills**: The candidate has a strong foundation in programming languages (C, Java, Python, R), web development (JavaScript, HTML, CSS, ReactJS, NodeJS), and databases (MySQL, MongoDB).
2. **Project Experience**: The candidate has hands-on experience in developing multiple projects, including a full-stack MERN Resume Builder web app, a Java-based MBTI Personality Type Calculator, and a travel planning web application.
3. **Academic Achievements**: The candidate has an impressive academic record with a 9.35 CGPA in B.Tech and 10 CGPA in SSC education.
4. **Certifications**: The candidate has obtained certifications in JavaScript Essentials and Frontend Developer Certificate, demonstrating their commitment to continuous learning.
5. **Soft Skills**: The candidate has listed problem-solving skills, communication skills, and time management & prioritization as their soft skills, which are essential for a professional work en

In [ ]:
display(Markdown(final_state["summary"]))


✅ Analysis:
**Key Strengths:**

1. **Technical Skills**: The candidate has a strong foundation in programming languages (C, Java, Python, R), web development (JavaScript, HTML, CSS, ReactJS, NodeJS), and databases (MySQL, MongoDB).
2. **Project Experience**: The candidate has hands-on experience in developing multiple projects, including a full-stack MERN Resume Builder web app, a Java-based MBTI Personality Type Calculator, and a travel planning web application.
3. **Academic Achievements**: The candidate has an impressive academic record with a 9.35 CGPA in B.Tech and 10 CGPA in SSC education.
4. **Certifications**: The candidate has obtained certifications in JavaScript Essentials and Frontend Developer Certificate, demonstrating their commitment to continuous learning.
5. **Soft Skills**: The candidate has listed problem-solving skills, communication skills, and time management & prioritization as their soft skills, which are essential for a professional work environment.

**Key Weaknesses:**

1. **Lack of Relevant Internship Experience**: The candidate's resume does not mention any relevant internship experience, which may be a concern for some employers.
2. **Limited Achievements in Co-curricular Activities**: Although the candidate has participated in a few co-curricular activities, such as a software hackathon and design-a-thon, the achievements are not substantial or notable.
3. **No Clear Career Goals**: The candidate's career objective is generic and does not provide a clear direction or specific goals.
4. **No Quantifiable Achievements**: The candidate's achievements are not quantified, making it difficult to assess the impact of their projects and work.
5. **Resume Format**: The resume format is not very polished, and some sections, such as the "INTERESTS" section, seem out of place.

**Suggestions for Improvement:**

1. **Gain relevant internship experience**: Try to secure internships in relevant fields to gain hands-on experience and build a professional network.
2. **Highlight achievements**: Quantify achievements and provide specific examples of accomplishments in projects and co-curricular activities.
3. **Refine career goals**: Clearly define career goals and tailor the resume to specific job applications.
4. **Polish the resume format**: Use a more professional resume format and ensure that sections are well-organized and easy to read.

🔍 Gaps:
Based on the provided resume and job description, here are the gaps, missing skills, and mismatch areas:

**Gaps:**

1. **Experience:** The job requires 3+ years of professional experience in software engineering, but the resume doesn't mention the exact duration of experience.
2. **Education:** The job requires a Bachelor's degree in Computer Science, Software Engineering, or a related technical field, which is present in the resume. However, a Master's or PhD in CS or related fields is preferred, which is not mentioned.

**Missing Skills:**

1. **Machine Learning:** The job description mentions experience with machine learning, large language models, or AI infrastructure as a nice-to-have skill, which is not present in the resume.
2. **Google Cloud Platform (GCP):** Although the resume mentions experience with cloud infrastructure, it doesn't specifically mention GCP, which is preferred by the job.
3. **Containerization:** The job mentions experience with Docker, Kubernetes, and Istio, but the resume doesn't explicitly mention these technologies.

**Mismatch Areas:**

1. **Programming Languages:** The job requires strong proficiency in one or more of the following programming languages: C++, Java, Go, Python, TypeScript, or Kotlin. The resume mentions experience with Java, Python, and C, but not with the other languages.
2. **Backend Development:** The job requires experience with scalable backend frameworks like Spring Boot, Express.js, FastAPI, Django, or NestJS. The resume mentions experience with React, NodeJS, and JavaScript, but not with these specific backend frameworks.
3. **Data & Storage:** The job requires strong knowledge of SQL databases and NoSQL databases. The resume mentions experience with MySQL and MongoDB, which aligns with the job requirements.

**Additional Observations:**

1. **Certifications:** The resume mentions some certifications like JavaScript Essentials and Frontend Developer Certificate, but not directly related to the job requirements.
2. **Projects:** The resume showcases some projects like Resume-Builder, MBTI Personality Type Calculator, and Travel Planning for Families. These projects demonstrate the candidate's skills, but may not directly align with Google's job requirements.

Overall, while the candidate has a good foundation in programming and software development, there are some gaps and mismatch areas, particularly in terms of experience, specific skills, and education.

🚀 Final Resume:
Sreetha Reddy Kyasaram  
sreethareddy10@gmail.com | +91 8121088416 | LinkedIn: www.linkedin.com/in/sreethareddy02

**Summary:**
Highly motivated and detail-oriented B.Tech student with a strong foundation in programming languages, web development, and databases. Proficient in developing scalable software systems with a passion for learning and innovation. Seeking a challenging internship opportunity to apply skills, gain hands-on experience, and contribute to a globally recognized organization.

**Education:**

* **Bachelor of Technology, Information Technology**  
  VNR Vignana Jyothi Institute of Engineering and Technology, Hyderabad  
  CGPA: 9.35/10
* **Intermediate Education**  
  Sri Chaitanya jr College, Kondapur  
  Percentage: 98.9%
* **SSC Education**  
  Panchavati Vidyalaya, Mahabubnagar  
  CGPA: 10/10

**Technical Skills:**

* **Programming Languages:** C, Java, Python, R-programming
* **Web Development:** JavaScript, HTML, CSS, ReactJS, NodeJS
* **Databases:** MySQL, MongoDB
* **Design and Visualization Tools:** AUTOCAD, Power BI
* **Developer Tools:** VS Code, Eclipse

**Projects:**

* **Resume-Builder:** Developed a full-stack MERN Resume Builder web app with drag-and-drop customization and real-time preview. Integrated LinkedIn/GitHub import, PDF/DOCX export, and secure authentication.
* **MBTI Personality Type Calculator:** Created a Java-based personality assessment tool using the Myers-Briggs Type Indicator (MBTI) framework and OOP principles.
* **Travel Planning for Families:** Built a travel itinerary generator web application to help families efficiently plan and organize their trips.

**Achievements:**

* **SOFTWARE HACKATHON:** Participated in the software hackathon conducted by VNRVJIET's National Annual Technical Symposium CONVERGENCE2K23R.
* **VNR DESIGN-A-THON:** Participated in VNR DESIGN-A-THON 2025, organized by VNRVJIET.

**Certifications:**

* **JavaScript Essentials:** Cisco Networking Academy, March 2025
* **Frontend Developer Certificate:** OneRoadMap, June 2025

**Soft Skills:**

* **Problem-solving Skills:** Proven ability to analyze and resolve complex technical issues.
* **Communication Skills:** Effective communication and teamwork skills.
* **Time Management & Prioritization:** Strong organizational skills to manage multiple tasks and deadlines.

**Relevant Coursework:**

* **Data Structures and Algorithms:** Implemented various data structures and algorithms in Java and Python.
* **Computer Networks:** Designed and simulated network protocols using NS-2.
* **Database Management Systems:** Developed and managed databases using MySQL and MongoDB.

**References:**
Available upon request.

🧠 Job Suggestions:
- Error on "{\rtf1\ansi" when compiling C program - Stack Overflow
  🔗 https://stackoverflow.com/questions/23669037/error-on-rtf1-ansi-when-compiling-c-program
- Textedit formatting issues : r/osx - Reddit
  🔗 https://www.reddit.com/r/osx/comments/6377mj/textedit_formatting_issues/
- Career Opportunities in Standards
  🔗 https://www.ansi.org/about/standards-careers
- github.gg
  🔗 https://www.github.gg/Ilovecatz17/remote-model-access
